In [1]:
import io
import json
import rich
import time
from openai import OpenAI

In [2]:
# Initialize client
client = OpenAI(base_url="http://ogxserver-service.llama.svc.cluster.local:8321/v1", api_key="fake")

In [3]:
# List available models
models = client.models.list()

# Extract LLM model details
llm_model = next(
    m for m in models.data
    if m.custom_metadata.get("model_type") == "llm"
)

model_id = llm_model.id
print(f"LLM Model: {model_id}")

LLM Model: vllm-inference/llama-32-3b-instruct


In [4]:
# 1. Define your payload using the model variable
batch_payload = (
    f'{{"custom_id": "req-001", "method": "POST", "url": "/v1/chat/completions", "body": {{"model": "{model_id}", "messages": [{{"role": "user", "content": "Explain metadata filtering."}}]}}}}\n'
    f'{{"custom_id": "req-002", "method": "POST", "url": "/v1/chat/completions", "body": {{"model": "{model_id}", "messages": [{{"role": "user", "content": "Explain hybrid search."}}]}}}}\n'
    f'{{"custom_id": "req-003", "method": "POST", "url": "/v1/chat/completions", "body": {{"model": "{model_id}", "messages": [{{"role": "user", "content": "Explain neural reranking."}}]}}}}\n'
)

# 2. Convert the payload into a raw bytes stream
new_file_stream = io.BytesIO(batch_payload.encode("utf-8"))

print("Uploading a fresh batch file to the storage backend...")
new_file = client.files.create(
    file=("live-demo-tasks.jsonl", new_file_stream),
    purpose="batch"
)
print(f" -> Generated New File ID: {new_file.id}")

# 3. Dispatch the first batch job
print("\nDispatching the batch processing request using the new file...")
batch_job = client.batches.create(
    input_file_id=new_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h"
)
print(f" -> Successfully Created Batch ID: {batch_job.id}")
print(f" -> Initial Processing Status: {batch_job.status}")

# 4. Cancel the active batch job
print("\nCanceling the batch...")
canceled_batch = client.batches.cancel(batch_job.id)
print(f" -> Updated Status: {canceled_batch.status}")

# Give the server a brief moment to cycle the state change internally
time.sleep(2)

# 5. Re-create and dispatch a brand new batch job using the exact same file
print("\nCreating a new batch...")
batch_job = client.batches.create(
    input_file_id=new_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h"
)

print(f" -> New Batch ID: {batch_job.id}")
print(f" -> New Status: {batch_job.status}")


Uploading a fresh batch file to the storage backend...
 -> Generated New File ID: file-78ac37c2202b476e88d310636fc87a55

Dispatching the batch processing request using the new file...
 -> Successfully Created Batch ID: batch_22a58dca0bd244f6
 -> Initial Processing Status: validating

Canceling the batch...
 -> Updated Status: cancelling

Creating a new batch...
 -> New Batch ID: batch_be492fdda3c348d5
 -> New Status: validating


In [5]:
rich.print(batch_job)

Batch(
    id='batch_be492fdda3c348d5',
    completion_window='24h',
    created_at=1781526334,
    endpoint='/v1/chat/completions',
    input_file_id='file-78ac37c2202b476e88d310636fc87a55',
    object='batch',
    status='validating',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=None,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=None,
    failed_at=None,
    finalizing_at=None,
    in_progress_at=None,
    metadata=None,
    model=None,
    output_file_id=None,
    request_counts=None,
    usage=None
)

In [6]:
batches = client.batches.list()
for batch in batches:
    print(batch.id, batch.status)

batch_be492fdda3c348d5 in_progress
batch_22a58dca0bd244f6 cancelled
batch_73342ba907594046 completed
batch_8c10a2cd19d742d1 cancelled
batch_748f544db1dc4aad cancelled
batch_38444f6f1c3645be completed


In [7]:
batch = client.batches.retrieve(batch_job.id)
rich.print(batch)

Batch(
    id='batch_be492fdda3c348d5',
    completion_window='24h',
    created_at=1781526334,
    endpoint='/v1/chat/completions',
    input_file_id='file-78ac37c2202b476e88d310636fc87a55',
    object='batch',
    status='in_progress',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=None,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=None,
    failed_at=None,
    finalizing_at=None,
    in_progress_at=None,
    metadata=None,
    model=None,
    output_file_id=None,
    request_counts=BatchRequestCounts(completed=0, failed=0, total=3),
    usage=None
)

In [8]:
print(f"\n[INFO] Starting execution monitoring loop for Batch: {batch_job.id}")
print("-" * 70)

while True:
    # 1. Retrieve the latest state from the server
    batch = client.batches.retrieve(batch_job.id)
    
    # 2. Print the beautifully formatted rich payload
    rich.print(batch)
    
    # 3. Check tracking progress if available
    if batch.request_counts:
        print(f"📊 Progress: {batch.request_counts.completed} / {batch.request_counts.total} requests completed.")
    
    # 4. Break the loop if it reaches a terminal status
    if batch.status in ["completed", "failed", "cancelled", "expired"]:
        print(f"\n🛑 Loop stopped. Final Batch Status: {batch.status.upper()}")
        break
        
    # 5. Wait for 10 seconds before pulling status again
    print("\n⏳ Sleeping for 10 seconds before next status check...\n")
    print("=" * 70)
    time.sleep(10)


[INFO] Starting execution monitoring loop for Batch: batch_be492fdda3c348d5
----------------------------------------------------------------------


Batch(
    id='batch_be492fdda3c348d5',
    completion_window='24h',
    created_at=1781526334,
    endpoint='/v1/chat/completions',
    input_file_id='file-78ac37c2202b476e88d310636fc87a55',
    object='batch',
    status='in_progress',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=None,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=None,
    failed_at=None,
    finalizing_at=None,
    in_progress_at=None,
    metadata=None,
    model=None,
    output_file_id=None,
    request_counts=BatchRequestCounts(completed=0, failed=0, total=3),
    usage=None
)

📊 Progress: 0 / 3 requests completed.

⏳ Sleeping for 10 seconds before next status check...



Batch(
    id='batch_be492fdda3c348d5',
    completion_window='24h',
    created_at=1781526334,
    endpoint='/v1/chat/completions',
    input_file_id='file-78ac37c2202b476e88d310636fc87a55',
    object='batch',
    status='in_progress',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=None,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=None,
    failed_at=None,
    finalizing_at=None,
    in_progress_at=None,
    metadata=None,
    model=None,
    output_file_id=None,
    request_counts=BatchRequestCounts(completed=0, failed=0, total=3),
    usage=None
)

📊 Progress: 0 / 3 requests completed.

⏳ Sleeping for 10 seconds before next status check...



...

📊 Progress: 0 / 3 requests completed.

⏳ Sleeping for 10 seconds before next status check...



Batch(
    id='batch_be492fdda3c348d5',
    completion_window='24h',
    created_at=1781526334,
    endpoint='/v1/chat/completions',
    input_file_id='file-78ac37c2202b476e88d310636fc87a55',
    object='batch',
    status='in_progress',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=None,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=None,
    failed_at=None,
    finalizing_at=None,
    in_progress_at=None,
    metadata=None,
    model=None,
    output_file_id=None,
    request_counts=BatchRequestCounts(completed=0, failed=0, total=3),
    usage=None
)

📊 Progress: 0 / 3 requests completed.

⏳ Sleeping for 10 seconds before next status check...



Batch(
    id='batch_be492fdda3c348d5',
    completion_window='24h',
    created_at=1781526334,
    endpoint='/v1/chat/completions',
    input_file_id='file-78ac37c2202b476e88d310636fc87a55',
    object='batch',
    status='completed',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=1781526368,
    error_file_id='file-7c7e6c05fa20479a89c1adc7f24b4639',
    errors=None,
    expired_at=None,
    expires_at=None,
    failed_at=None,
    finalizing_at=None,
    in_progress_at=None,
    metadata=None,
    model=None,
    output_file_id='file-914ec999d44a468899ee6f1e92b2fb69',
    request_counts=BatchRequestCounts(completed=3, failed=0, total=3),
    usage=None
)

📊 Progress: 3 / 3 requests completed.

🛑 Loop stopped. Final Batch Status: COMPLETED


In [9]:
OUTPUT_FILE_ID=batch.output_file_id

print("[INFO] Fetching batch contents...")
raw_content = client.files.content(OUTPUT_FILE_ID).text

print("\n=== CLEAN PARSED TEXT ===")
print("-" * 60)

# Split by lines and filter out empty items
for line in raw_content.strip().split("\n"):
    if not line.strip():
        continue
        
    # 1. Parse the main line object
    data = json.loads(line)
    custom_id = data.get("custom_id")
    
    # 2. Get the response envelope
    response_data = data.get("response", {})
    body_data = response_data.get("body", {})
    
    if isinstance(body_data, str):
        body_data = json.loads(body_data)
        
    # 3. Safely grab choices now that body_data is a dictionary
    choices = body_data.get("choices", [])
    
    if choices:
        content = choices[0].get("message", {}).get("content", "")
        print(f"🆔 Task:   {custom_id}")
        print(f"🤖 Answer: {content.strip()}")
        print("-" * 60)

[INFO] Fetching batch contents...

=== CLEAN PARSED TEXT ===
------------------------------------------------------------
🆔 Task:   req-001
🤖 Answer: Metadata filtering is a process used to manage and control the flow of data by applying rules to metadata, which is data that describes other data. Metadata filtering is commonly used in various applications, including data storage, data exchange, and data security.

Metadata filtering involves analyzing and processing metadata to determine whether it meets certain criteria or rules. This can include filtering out metadata that is sensitive, confidential, or irrelevant to
------------------------------------------------------------
🆔 Task:   req-002
🤖 Answer: Hybrid Search

Hybrid search is a search algorithm that combines the strengths of multiple search algorithms to achieve better performance and efficiency. It is a technique used in information retrieval and search engines to improve the search results by leveraging the advantages of 